# NB1 — Delta Lake Basics (lightweight path)

**Stack:** `deltalake` (delta-rs) + Polars + DuckDB. No Spark, no JVM.
Maps to slide §2 (Delta Lake) + deliverable bullet 1.

> Spark equivalent: `spark.read.format("delta").load(path)` ↔ `DeltaTable(path).to_pyarrow_table()`.
> Same on-disk format, different binding.

In [1]:
import sys, platform, importlib.metadata
from pathlib import Path
repo_root = next(p for p in (Path.cwd(), *Path.cwd().parents)
                 if (p / "scripts/lakehouse.py").exists() and (p / "notebooks/_setup.py").exists())
sys.path.insert(0, str(repo_root / "notebooks"))
print("Python:", sys.version.split()[0], "|", sys.executable)
print("OS:", platform.platform())
print("Libraries:", {name: importlib.metadata.version(name) for name in
    ("deltalake", "pyiceberg", "duckdb", "polars", "pyarrow", "numpy")})


Python: 3.11.3 | D:\AI_Thucchien\K4-Track02-Day18-HoangThaiDat-2A202602959-Lakehouse-Lab\.venv\Scripts\python.exe
OS: Windows-10-10.0.26200-SP0
Libraries: {'deltalake': '1.6.6', 'pyiceberg': '0.12.0', 'duckdb': '1.5.6', 'polars': '1.44.2', 'pyarrow': '25.0.1', 'numpy': '2.4.6'}


In [2]:
import _setup  # noqa: F401  -- adds scripts/ to sys.path (file-relative)
import polars as pl
from deltalake import DeltaTable, write_deltalake
from lakehouse import path, reset

table_path = path("scratch", "users_delta")
reset(table_path)  # idempotent rerun

## 1. Write a Delta table

In [3]:
df = pl.DataFrame({
    "id": [1, 2, 3],
    "name": ["alice", "bob", "charlie"],
    "age": [30, 25, 35],
    "city": ["Hanoi", "HCMC", "Danang"],
})
write_deltalake(table_path, df.to_arrow(), mode="overwrite")

## 2. Read it back + inspect transaction log

Look at `_lakehouse/scratch/users_delta/_delta_log/00000000000000000000.json` —
that's the transaction log. Same JSON format Spark/Databricks would write.

In [4]:
dt = DeltaTable(table_path)
print(pl.from_arrow(dt.to_pyarrow_table()))
print("\nHistory:")
for h in dt.history():
    print(f"  v{h['version']}  {h['operation']}  {h.get('operationMetrics', {})}")

shape: (3, 4)
┌─────┬─────────┬─────┬────────┐
│ id  ┆ name    ┆ age ┆ city   │
│ --- ┆ ---     ┆ --- ┆ ---    │
│ i64 ┆ str     ┆ i64 ┆ str    │
╞═════╪═════════╪═════╪════════╡
│ 1   ┆ alice   ┆ 30  ┆ Hanoi  │
│ 2   ┆ bob     ┆ 25  ┆ HCMC   │
│ 3   ┆ charlie ┆ 35  ┆ Danang │
└─────┴─────────┴─────┴────────┘

History:
  v0  WRITE  {'num_added_files': 1, 'num_removed_files': 0, 'num_partitions': 0, 'num_added_rows': 3, 'execution_time_ms': 4, 'num_retries': 0}


## 3. Schema enforcement — try to write a wrong schema

In [5]:
bad = pl.DataFrame({"id": [4], "name": ["dan"], "age": ["thirty"], "city": ["Hue"]})
schema_enforcement_blocked = False
version_before_bad_write = DeltaTable(table_path).version()
rows_before_bad_write = DeltaTable(table_path).count()
try:
    write_deltalake(table_path, bad.to_arrow(), mode="append")
    print("UNEXPECTED: bad write succeeded — schema enforcement broken")
except Exception as e:
    schema_enforcement_blocked = True
    msg = str(e).splitlines()[0][:120]
    print(f"BLOCKED by schema enforcement (expected): {type(e).__name__}: {msg}")

assert schema_enforcement_blocked, "The invalid age=str write must fail"
failed_write_version_unchanged = DeltaTable(table_path).version() == version_before_bad_write
assert failed_write_version_unchanged
assert DeltaTable(table_path).count() == rows_before_bad_write
print("Failed write preserved the table version and row count.")

BLOCKED by schema enforcement (expected): Exception: Cast error: Cannot cast string 'thirty' to value of Int64 type
Failed write preserved the table version and row count.


## 4. Schema evolution (opt-in)

In [6]:
new = pl.DataFrame({
    "id": [4], "name": ["dan"], "age": [28], "city": ["Hue"], "tier": ["premium"],
})
write_deltalake(table_path, new.to_arrow(), mode="append", schema_mode="merge")
dt = DeltaTable(table_path)
# Sort by id so the printout is stable across reruns — Delta does not
# preserve write-order across appends.
print(pl.from_arrow(dt.to_pyarrow_table()).sort("id"))

shape: (4, 5)
┌─────┬─────────┬─────┬────────┬─────────┐
│ id  ┆ name    ┆ age ┆ city   ┆ tier    │
│ --- ┆ ---     ┆ --- ┆ ---    ┆ ---     │
│ i64 ┆ str     ┆ i64 ┆ str    ┆ str     │
╞═════╪═════════╪═════╪════════╪═════════╡
│ 1   ┆ alice   ┆ 30  ┆ Hanoi  ┆ null    │
│ 2   ┆ bob     ┆ 25  ┆ HCMC   ┆ null    │
│ 3   ┆ charlie ┆ 35  ┆ Danang ┆ null    │
│ 4   ┆ dan     ┆ 28  ┆ Hue    ┆ premium │
└─────┴─────────┴─────┴────────┴─────────┘


## 5. Query with DuckDB via Arrow (part of the required notebook)

In [7]:
import duckdb

# We hand DuckDB an Arrow table rather than calling `delta_scan()`. delta_scan
# autoloads a DuckDB extension over the network — fine at home, a support
# ticket in a firewalled classroom. Arrow registration is zero-copy and offline.
con = duckdb.connect()
con.register("users", DeltaTable(table_path).to_pyarrow_table())
tier_counts = con.sql("SELECT tier, count(*) AS n FROM users GROUP BY 1 ORDER BY 1").fetchall()
print(tier_counts)

[('premium', 1), (None, 3)]


## ✅ Deliverable check
- [ ] `_delta_log/` contains JSON files
- [ ] Schema enforcement blocked the bad write
- [ ] schema_mode="merge" added the `tier` column
- [ ] DuckDB query returned 2 tier groups
The enforcement check records the actual exception and verifies that the
failed write did not commit a version or add rows.

In [8]:
from pathlib import Path as _Path  # noqa: E402

_log = sorted(_Path(table_path).glob("_delta_log/*.json"))
_cols = DeltaTable(table_path).schema().to_arrow().names
print("Transaction log commits:")
for commit in _log:
    print(f"  {commit.name}")
print(f"\nFirst commit JSON ({_log[0].name}):")
print(_log[0].read_text(encoding="utf-8"))
checks = {
    "_delta_log/ has JSON commits": len(_log) >= 2,
    "schema enforcement blocked bad write": schema_enforcement_blocked,
    "tier column added via schema_mode=merge": "tier" in _cols,
    "duckdb sees 2 tier groups": len(tier_counts) == 2,
}
for k, v in checks.items():
    print(f"  [{'PASS' if v else 'FAIL'}] {k}")
assert all(checks.values()), "NB1 incomplete — see FAIL rows above"
print("\nNB1 complete.")

Transaction log commits:
  00000000000000000000.json
  00000000000000000001.json

First commit JSON (00000000000000000000.json):
{"commitInfo":{"timestamp":1791170216318,"operation":"WRITE","operationParameters":{"mode":"Overwrite"},"engineInfo":"delta-rs:py-1.6.6","clientVersion":"delta-rs.py-1.6.6","operationMetrics":{"num_added_files":1,"num_removed_files":0,"num_partitions":0,"num_added_rows":3,"execution_time_ms":4,"num_retries":0}}}
{"protocol":{"minReaderVersion":1,"minWriterVersion":2}}
{"metaData":{"id":"ca94c270-39a6-46aa-ba56-204251fe9b79","name":null,"description":null,"format":{"provider":"parquet","options":{}},"schemaString":"{\"type\":\"struct\",\"fields\":[{\"name\":\"id\",\"type\":\"long\",\"nullable\":true,\"metadata\":{}},{\"name\":\"name\",\"type\":\"string\",\"nullable\":true,\"metadata\":{}},{\"name\":\"age\",\"type\":\"long\",\"nullable\":true,\"metadata\":{}},{\"name\":\"city\",\"type\":\"string\",\"nullable\":true,\"metadata\":{}}]}","partitionColumns":[],"cre

## Bằng chứng thu thập từ lần chạy này

Các giá trị dưới đây lấy trực tiếp từ biến và bảng đã thực thi ở trên.

In [9]:
import json
lab_metrics = {
        "json_commits": len(_log), "schema_enforcement_blocked": schema_enforcement_blocked,
        "failed_write_version_unchanged": failed_write_version_unchanged,
        "tier_present": "tier" in _cols, "tier_groups": len(tier_counts),
        "rows_after_evolution": DeltaTable(table_path).count()
    }
print("Bằng chứng rubric — NB01")
for label, value in lab_metrics.items():
    print(f"  {label}: {value}")
print('LAB_METRICS_JSON=' + json.dumps(lab_metrics, ensure_ascii=False, sort_keys=True))


Bằng chứng rubric — NB01
  json_commits: 2
  schema_enforcement_blocked: True
  failed_write_version_unchanged: True
  tier_present: True
  tier_groups: 2
  rows_after_evolution: 4
LAB_METRICS_JSON={"failed_write_version_unchanged": true, "json_commits": 2, "rows_after_evolution": 4, "schema_enforcement_blocked": true, "tier_groups": 2, "tier_present": true}


## Giải thích kết quả và giới hạn phép đo

Có 2 commit JSON. Ghi `age='thirty'` bị chặn và không thay đổi version/số dòng; opt-in schema merge thêm `tier`, DuckDB thấy 2 nhóm (bao gồm NULL ở dữ liệu cũ). Transaction log xác định các file thuộc trạng thái bảng đã commit.